# Outlines
* Required Installation
* Mathematical Background
  * Additive Model 
  * Uniqueness
  * Reliability
  * Uniformity
  * Modelling Attack: Logistic Regression
* Classical Arbiter PUF
  * Additive Delay Model
  * PyPUF Implementation
  * Security Analysis

## Required Installation

Run the cell below once to install the required packages. (Drop `--break-system-packages` if you're running inside a virtual environment — it's only needed on systems with an "externally managed" Python installation, e.g. recent Debian/Ubuntu.)

In [1]:
%pip install pypuf --break-system-packages --quiet
%pip install scikit-learn --break-system-packages --quiet
%pip install numpy --break-system-packages --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression

from pypuf.simulation import ArbiterPUF
from pypuf.io import random_inputs
from pypuf.metrics import reliability, uniqueness, similarity_data

np.set_printoptions(precision=4, suppress=True)

## 1. Mathematical Background
### 1.1. Additive Delay Model
The Additive Delay Model is the general framework used across the whole family of **delay-based PUFs**.

**Core idea:** the physical quantity that decides the output (delay difference, phase difference, etc.) is a **linear combination** of a challenge-derived feature vector and a chip-specific weight vector:

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0, \qquad r = \text{sign}(-\Delta)$$

- $\mathbf{w}$: fixed but unknown, chip-specific weights (encodes physical delay/phase variations)
- $\boldsymbol{\phi}(\mathbf{c})$: an **architecture-specific** transform of the challenge
- $w_0$: bias term

**Why this abstraction matters:**
1. **ML attack analysis generalizes** — any PUF fitting $\Delta = \mathbf{w}^T\phi+w_0$ is vulnerable to the same attack class (logistic regression, SVM), regardless of architecture.
2. **Security comparison reduces to one question:** how non-linear can $\phi(\mathbf{c})$ (or $\Delta$'s dependence on it) be made? XOR-PUF and Feed-Forward PUF are attempts to break this linearity without abandoning the additive skeleton.

**In short:** the Additive Delay Model abstracts any delay-based PUF as $\Delta = \mathbf{w}^T\phi(\mathbf{c})+w_0,\ r=\text{sign}(-\Delta)$; APUF is the special case where $\phi$ is the simple parity vector, while other architectures either redefine $\phi$ or combine multiple additive components non-linearly (e.g. via XOR).

### 1.2. Reliability
**Definition:** Measures how stable a single PUF instance's responses are when the same challenge is queried multiple times under simulated noise.

**Theoretical formula**:

$$\text{Reliability} = \Pr_{\text{eval}}\left[\text{eval}(x) = \text{eval}(x)\right]$$

That is, the probability that the same challenge $x$, evaluated twice under the noisy evaluation process, yields the same result.

In [3]:
# Reliability test.
#
# IMPORTANT: `instance` must have been created with `noisiness > 0`
# (e.g. ArbiterPUF(..., noisiness=0.05)). A noiseless instance
# (noisiness=0, the default) will always score a perfect 1.0, which is
# not informative.
#
# Returns the overall reliability score in [0, 1]. Ideal: 1.0.
def test_reliability(instance, seed: int = 10, N: int = 1000, r: int = 17,
                      verbose: bool = True) -> float:
    rel_per_challenge = reliability(instance, seed=seed, N=N, r=r)
    overall = float(np.average(rel_per_challenge))

    if verbose:
        print(f"[Reliability] Overall score : {overall:.4f}  (ideal: 1.0000)")
        print(f"[Reliability] Min observed  : {rel_per_challenge.min():.4f}")
        print(f"[Reliability] Max observed  : {rel_per_challenge.max():.4f}")

    return overall

### 1.3. Uniqueness

**Definition:** Measures how distinguishable a set of PUF instances (representing different physical chips of the same design) are from one another, based on their responses to the same set of random challenges.

**First, pairwise similarity between two chips is defined:**

$$\text{sim}(f,g) = \Pr_x\left[f(x) = g(x)\right] \approx \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[f(c_i) = g(c_i)]$$

**Uniqueness formula**, averaged over all chip pairs:

$$\text{Uniqueness} = 1 - 2 \cdot \mathbb{E}_{f \neq g}\left[\left|\tfrac{1}{2} - \text{sim}(f,g)\right|\right]$$

**Intuition:**
- Ideally, two different chips behave like independent coin flips on a random challenge, i.e. $\text{sim}(f,g) = 0.5$ → $|\tfrac{1}{2}-0.5|=0$ → Uniqueness $=1$.
- If two chips always agree ($\text{sim}=1$) or always disagree ($\text{sim}=0$), in both cases $|\tfrac{1}{2}-\text{sim}| = 0.5$ → Uniqueness $= 1 - 2(0.5) = 0$ (worst case: chips are indistinguishable).

For $l$ chips, the average is taken over all $\binom{l}{2}$ pairs.

In [4]:
# Uniqueness test
#
# `instances` must contain at least 2 PUF instances (e.g. created with
# different seeds, each representing a different "chip").
#
# Returns the overall uniqueness score in [0, 1]. Ideal: 1.0.
def test_uniqueness(instances: list, seed: int = 20, N: int = 2000,
                     verbose: bool = True) -> float:
    if len(instances) < 2:
        raise ValueError("Uniqueness test requires at least 2 PUF instances.")

    score = float(uniqueness(instances, seed=seed, N=N)[0])

    if verbose:
        print(f"[Uniqueness]  Score ({len(instances)} chips): {score:.4f}  "
              f"(ideal: 1.0000)")

    return score

### 1.4. Uniformity

**Definition:** Measures the balance of 0s and 1s in a single PUF instance's responses over a large random challenge set. Bias is measured by how far this value deviates from $0.5$.

$$\text{Uniformity} = \frac{1}{N}\sum_{i=1}^{N} r_i \quad, \quad r_i \in \{0,1\}$$



In [5]:
# Uniformity test.
#
# pypuf  has no function to calculate for this, so it is 
# computed directly: the fraction of responses equal
# to 1 (pypuf encodes responses as -1/+1).
#
# Returns the fraction of responses equal to 1. Ideal: 0.5.

def test_uniformity(instance, n: int, N: int = 5000, seed: int = 30,
                     verbose: bool = True) -> float:
    challenges = random_inputs(n=n, N=N, seed=seed)
    responses = instance.eval(challenges)  # values are -1 / +1

    responses_binary = (responses == 1).astype(int)
    score = float(responses_binary.mean())

    if verbose:
        deviation = abs(score - 0.5)
        print(f"[Uniformity]  Score: {score:.4f}  (ideal: 0.5000)  "
              f"-> {score*100:.2f}% ones / {(1-score)*100:.2f}% zeros, "
              f"deviation: {deviation:.4f}")

    return score

### 1.5. Modeling Attack (Logistic Regression Attack)

**Source of the weakness — additive delay model:**

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0$$

Response:
$$r = \text{sign}(-\Delta) = \begin{cases}1 & \Delta<0\\0 & \Delta\geq0\end{cases}$$

**Attacker's goal:** without knowing the true $\mathbf{w}$, estimate $\hat{\mathbf{w}}$ from a set of $(\mathbf{c}_i, r_i)$ pairs.

**The logistic regression model** directly models the sign of $\Delta$ via a sigmoid function:

$$P(r=1 \mid \boldsymbol{\phi}) = \sigma(\hat{\mathbf{w}}^T \boldsymbol{\phi} + \hat{w}_0) = \frac{1}{1+e^{-(\hat{\mathbf{w}}^T \boldsymbol{\phi}+\hat{w}_0)}}$$

**Training minimizes the cross-entropy (log-loss):**

$$\mathcal{L}(\hat{\mathbf{w}}) = -\frac{1}{N}\sum_{i=1}^{N}\Big[r_i \log P(\boldsymbol{\phi}_i) + (1-r_i)\log\big(1-P(\boldsymbol{\phi}_i)\big)\Big] + \lambda\|\hat{\mathbf{w}}\|_2^2$$

(scikit-learn's `LogisticRegression` parameter `C` corresponds to $\lambda = 1/C$ — the regularization term.)

**Why it works so well:** since $\Delta$ is a **linear** function of $\boldsymbol{\phi}(\mathbf{c})$, logistic regression's decision boundary ($\hat{\mathbf{w}}^T\boldsymbol{\phi}+\hat{w}_0 = 0$) belongs to the **exact same hypothesis class** as the PUF's true decision boundary ($\mathbf{w}^T\boldsymbol{\phi}+w_0=0$). This is why gradient descent can converge to $\hat{\mathbf{w}} \approx c\cdot\mathbf{w}$ (for some positive scaling constant $c$) with a finite number of CRPs.

**Attack success metric** (same formula as the similarity function):
$$\text{Attack Accuracy} = \frac{1}{N_{\text{test}}}\sum_{i=1}^{N_{\text{test}}} \mathbb{1}\big[\hat{r}_i = r_i\big]$$


In [6]:
# Converts raw -1/+1 challenges into parity ("phi") feature vectors,
# the linear feature space in which a classical Arbiter PUF's additive
# delay model becomes a simple dot product.
#
#     phi_i = c_i * c_(i+1) * ... * c_n   (cumulative product from the right)
def to_parity_vectors(challenges: np.ndarray) -> np.ndarray:
    reversed_cumprod = np.cumprod(challenges[:, ::-1], axis=1)
    return reversed_cumprod[:, ::-1]

# Logistic Regression modeling attack.
#
# Collects N challenge-response pairs, trains a logistic regression
# classifier on the parity-transformed challenges, and evaluates both
# train/test accuracy and similarity on a completely unseen challenge set.
#
# Returns a dict with keys: 'train_accuracy', 'test_accuracy', 'fresh_similarity'
def run_ml_attack(instance, n: int, N: int = 5000, split_ratio: float = 0.8,
                   train_seed: int = 2, fresh_seed: int = 5,
                   fresh_N: int = 2000, verbose: bool = True) -> dict:
    challenges = random_inputs(n=n, N=N, seed=train_seed)
    responses = instance.eval(challenges)

    split = int(N * split_ratio)
    X_train_raw, X_test_raw = challenges[:split], challenges[split:]
    y_train, y_test = responses[:split], responses[split:]

    y_train_bin = (y_train == 1).astype(int)
    y_test_bin = (y_test == 1).astype(int)

    Phi_train = to_parity_vectors(X_train_raw)
    Phi_test = to_parity_vectors(X_test_raw)

    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(Phi_train, y_train_bin)

    train_acc = clf.score(Phi_train, y_train_bin)
    test_acc = clf.score(Phi_test, y_test_bin)

    fresh_challenges = random_inputs(n=n, N=fresh_N, seed=fresh_seed)
    true_responses = instance.eval(fresh_challenges)
    phi_fresh = to_parity_vectors(fresh_challenges)
    pred_responses = np.where(clf.predict(phi_fresh) == 1, 1, -1)
    fresh_similarity = float(np.ravel(
        similarity_data(true_responses, pred_responses))[0])

    if verbose:
        print(f"[ML Attack]   Train accuracy      : {train_acc:.4f}")
        print(f"[ML Attack]   Test accuracy       : {test_acc:.4f}")
        print(f"[ML Attack]   Unseen-CRP similarity: {fresh_similarity:.4f}")

    return {
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "fresh_similarity": fresh_similarity,
    }

## 2. Classical Arbiter PUF (A-PUF)
<!--
Source - https://stackoverflow.com/a/52003495
Posted by Roman Vogt, modified by community. See post 'Timeline' for change history
Retrieved 2026-08-29, License - CC BY-SA 4.0
-->

![image info](./Pictures/arbiter-puf.png)

### 2.1. Additive Delay Mode
Let $\Delta_{i-1}$ be the delay difference entering stage $i$. The new difference $\Delta_i$ leaving that stage depends on the challenge bit $c_i$:

- **$c_i=0$ (straight):** signals don't swap → $\Delta_i = \Delta_{i-1} + \delta_i^{0}$
- **$c_i=1$ (crossed):** signals swap, so the previous difference's **sign flips** (the leading signal is now on the other line) → $\Delta_i = -\Delta_{i-1} + \delta_i^{1}$

Both cases combine into a single formula:

$$\Delta_i = (1-2c_i)\,\Delta_{i-1} + \delta_i(c_i), \qquad \delta_i(c_i) = (1-2c_i)\alpha_i + \beta_i$$

where:
- $\alpha_i$: the "straight-through" delay-difference component of that stage (chip-specific, due to manufacturing variation)
- $\beta_i$: a fixed delay component introduced by the crossover itself

tarting from $\Delta_0 = 0$ and unrolling for $i=1,\dots,n$:

$$\Delta_n = \sum_{i=1}^{n} \delta_i(c_i) \cdot \prod_{j=i+1}^{n}(1-2c_j)$$

Each stage's contribution ($\delta_i$) enters the sum multiplied by a coefficient that depends on **how many times all subsequent stages flip the sign**. This "product of subsequent stages" term is exactly the **parity vector**:

$$\phi_i = \prod_{j=i}^{n} (1-2c_j)$$

After algebraic simplification (grouping the $\alpha_i$ terms with $\phi_i$, and collapsing all $\beta_i$ terms into a single bias):

$$\boxed{\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0 = \sum_{i=1}^{n} \alpha_i \phi_i + w_0}$$

$$r = \begin{cases} 1 & \Delta < 0 \\ 0 & \Delta \geq 0 \end{cases}$$

where:
- $\mathbf{w}=(\alpha_1,\dots,\alpha_n)$: a chip-specific weight vector, randomly fixed at manufacturing time
- $\boldsymbol{\phi}(\mathbf{c})$: the architecture-specific parity vector computed from the challenge
- $w_0$: a constant bias term (sum of all $\beta_i$ contributions)

### 2.2. PyPUF Implementation
Create a 64-bit classical Arbiter PUF and generate a batch of challenge-response pairs (CRPs).

In [10]:
n = 64  # challenge length / number of MUX stages

puf = ArbiterPUF(n=n, seed=1)
print(f"[+] Simulated a {n}-bit classical Arbiter PUF (seed=1)")

challenges = random_inputs(n=n, N=5000, seed=2)
responses = puf.eval(challenges)
print(f"[+] Generated {len(challenges)} challenge-response pairs")
print(f"    Example challenge (first 10 bits): {challenges[0][:10]}")
print(f"    Corresponding response           : {responses[0]}")

[+] Simulated a 64-bit classical Arbiter PUF (seed=1)
[+] Generated 5000 challenge-response pairs
    Example challenge (first 10 bits): [-1 -1  1  1  1 -1  1  1  1  1]
    Corresponding response           : -1


## 2.3. Security Analysis
### 2.3.1. Reliability 

In [11]:
# Reliability requires a NOISY instance (same seed = same "chip",
# but with simulated environmental noise turned on)
noisy_puf = ArbiterPUF(n=n, seed=1, noisiness=0.05)
test_reliability(noisy_puf, seed=10, N=1000, r=17)

[Reliability] Overall score : 0.9651  (ideal: 1.0000)
[Reliability] Min observed  : 0.0588
[Reliability] Max observed  : 1.0000


0.9650588235294119

### 2.3.2. Uniqueness

In [12]:
# Uniqueness requires MULTIPLE chip instances (different seeds)
chip_instances = [ArbiterPUF(n=n, seed=s) for s in range(100, 110)]
test_uniqueness(chip_instances, seed=20, N=2000)

[Uniqueness]  Score (10 chips): 0.9441  (ideal: 1.0000)


0.9440666666666666

### 2.3.3. Uniqueness

In [13]:
# Uniformity is evaluated on a single chip instance
test_uniformity(puf, n=n, N=5000, seed=30)

[Uniformity]  Score: 0.5050  (ideal: 0.5000)  -> 50.50% ones / 49.50% zeros, deviation: 0.0050


0.505

### 2.3.4. Modelling Attack: Logistic Regression

In [14]:
run_ml_attack(puf, n=n, N=5000)

[ML Attack]   Train accuracy      : 0.9962
[ML Attack]   Test accuracy       : 0.9840
[ML Attack]   Unseen-CRP similarity: 0.9880


{'train_accuracy': 0.99625, 'test_accuracy': 0.984, 'fresh_similarity': 0.988}